# Image Description and Face Identification Pipeline
This notebook implements a pipeline that:

Loads known face encodings from known_faces/ directory
Processes images in images/ directory
Uses deterministic face recognition (face_recognition) to identify people
Uses a local vision-capable LLM (via pyautogen) to generate natural-language image descriptions
Saves each description as a Markdown (.md) file with the same base name as the image
Core Principle: The tool identify_people_in_image is the source of truth for person identification. The LLM is a consumer and narrator, not a classifier.

In [ ]:
# Suppress deprecation warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='face_recognition_models')
warnings.filterwarnings('ignore', category=DeprecationWarning)
print('Warnings filtered.')


In [ ]:
# Check and optionally install required packages
import sys
import subprocess
import importlib

required_packages = [
    ('face_recognition', 'face_recognition'),
    #('pyautogen', 'autogen'),  # or autogen_ext
    ('Pillow', 'PIL'),
    ('opencv-python', 'cv2'),
    ('numpy', 'numpy'),
]

def check_package(pip_name, import_name):
    try:
        importlib.import_module(import_name)
        return True
    except ImportError:
        return False

missing = []
for pip_name, import_name in required_packages:
    if not check_package(pip_name, import_name):
        missing.append(pip_name)

if missing:
    print(f"Missing packages: {', '.join(missing)}")
    print("You can install them with:")
    print(f"  pip install {' '.join(missing)}")
    print("Or run:")
    print("  pip install -r requirements.txt")
    print()
    # Uncomment the line below to auto-install missing packages
    # subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
else:
    print("All required packages are installed.")


In [ ]:
# Import required libraries
import sys
import os
import json
from pathlib import Path
from datetime import datetime

# Add src directory to Python path
src_path = Path.cwd() / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    from face_recognition_tool import load_known_faces, identify_people_in_image
    from tools import identify_people_in_image_tool, get_tool_schema
    from vision_client import get_vision_model_client, describe_image_with_identification, describe_with_direct_ollama
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure you have installed the required packages:")
    print("  face_recognition, pyautogen, Pillow, opencv-python, numpy")
    print("Also ensure the src/ directory contains the modules.")


In [ ]:
# Configuration
KNOWN_FACES_DIR = "Swansea 2026/Swansea_Staff"

IMAGES_DIR = "Swansea 2026"
OUTPUT_DIR = Path("Swansea 2026/annotated")

# Create directories if they don't exist
for d in [KNOWN_FACES_DIR, IMAGES_DIR, OUTPUT_DIR]:
    Path(d).mkdir(exist_ok=True)
    print(f"Directory {d}: {Path(d).resolve()}")

# Load known face encodings
print("Loading known faces...")
load_known_faces(KNOWN_FACES_DIR)
print("Known faces loaded.")

In [ ]:
KNOWN_FACES_DIR = "WarnersBayYear12/faces"

IMAGES_DIR = "WarnersBayYear12"
OUTPUT_DIR = Path("WarnersBayYear12/annotated")
# Load known face encodings
print("Loading known faces...")
load_known_faces(KNOWN_FACES_DIR)
print("Known faces loaded.")


In [ ]:
from pathlib import Path
import shutil

# CHANGE THIS to your image directory
TARGET_DIR = Path("Swansea 2026/Swansea_Staff")

print("Working directory:", Path.cwd())
print("Target directory:", TARGET_DIR.resolve())

if not TARGET_DIR.exists():
    raise RuntimeError("Target directory does not exist")

images = [p for p in TARGET_DIR.iterdir() if p.suffix.lower() == ".jpg"]

print(f"Found {len(images)} JPG images")

if not images:
    raise RuntimeError("No JPG images found — nothing to do")

for img in images:
    folder_name = img.stem  # filename without extension
    folder_path = TARGET_DIR / folder_name

    # Create folder if needed
    folder_path.mkdir(exist_ok=True)

    dest = folder_path / img.name

    if dest.exists():
        print(f"SKIP (already exists): {dest}")
        continue

    print(f"Moving: {img.name} -> {folder_path}/")
    shutil.move(str(img), str(dest))

print("Done.")


In [ ]:
##################
## IMAGE RESIZE ##
##################

from pathlib import Path
from PIL import Image

# CHANGE THIS to the top-level folder
ROOT_DIR = Path("Swansea 2026/Swansea_Staff")

SCALE_FACTOR = 0.8
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

print("Root directory:", ROOT_DIR.resolve())

if not ROOT_DIR.exists():
    raise RuntimeError("Root directory does not exist")

images = [p for p in ROOT_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

print(f"Found {len(images)} image(s)")

if not images:
    raise RuntimeError("No images found — nothing to resize")

for img_path in images:
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            new_size = (int(w * SCALE_FACTOR), int(h * SCALE_FACTOR))

            resized = img.resize(new_size, Image.LANCZOS)
            resized.save(img_path)

        print(f"Resized: {img_path} ({w}x{h} → {new_size[0]}x{new_size[1]})")

    except Exception as e:
        print(f"FAILED: {img_path} ({e})")

print("Done.")


In [ ]:
import json
from pathlib import Path
import cv2
import face_recognition
from PIL import Image, ImageDraw, ImageFont

# Make sure OUTPUT_DIR exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

# Optional font (fallback to default if not found)
try:
    FONT = ImageFont.truetype("arial.ttf", 16)
except:
    FONT = ImageFont.load_default()

# Collect images
image_files = [
    f for f in Path(IMAGES_DIR).iterdir()
    if f.suffix.lower() in IMAGE_EXTENSIONS
]

# Face recognition model: "hog" for CPU, "cnn" for GPU
FACE_MODEL = "cuda"

# Resize factor for memory efficiency
RESIZE_FACTOR = 0.33  # 33% size
# ----------------------------------------

print(f"Found {len(image_files)} image(s) to process.")

# Helper: compute center of a box
def center(box):
    top, right, bottom, left = box
    return ((left + right) // 2, (top + bottom) // 2)

for img_path in image_files:
    print(f"\n--- Processing {img_path.name} ---")

    # Load image
    image = face_recognition.load_image_file(img_path)

    # Optional: resize for memory if CNN
    small = cv2.resize(
        image,
        None,
        fx=RESIZE_FACTOR,
        fy=RESIZE_FACTOR,
        interpolation=cv2.INTER_AREA
    ) if FACE_MODEL == "cnn" else image

    # Detect face locations
    face_locations = face_recognition.face_locations(
        small,
        model=FACE_MODEL
    )

    # If resized, scale coordinates back to original
    if FACE_MODEL == "cnn" and RESIZE_FACTOR != 1.0:
        scale = 1 / RESIZE_FACTOR
        face_locations = [
            (
                int(top * scale),
                int(right * scale),
                int(bottom * scale),
                int(left * scale)
            )
            for (top, right, bottom, left) in face_locations
        ]

    print(f"Faces detected: {len(face_locations)}")

    # Identify people (external tool)
    ident_result = identify_people_in_image_tool(str(img_path))
    people = ident_result.get("people", [])

    print(f"People identified: {len(people)}")

    # Pair faces with people by index (works if tool preserves order)
    faces_with_people = list(zip(face_locations, people))

    # Sort left-to-right or row-wise (top → bottom, then left → right)
    # For top → bottom then left → right:
    faces_with_people.sort(key=lambda x: (x[0][0], x[0][3]))  # (top, left)

    # Prepare PIL image for drawing
    pil_image = Image.fromarray(image)
    draw = ImageDraw.Draw(pil_image)

    for idx, (box, person) in enumerate(faces_with_people):
        top, right, bottom, left = box
        name = person.get("name", "Unknown")

        # Determine box/text color
        if name == "Unknown":
            box_color = "red"
        else:
            box_color = "green"
        
        # Draw box
        draw.rectangle(((left, top), (right, bottom)), outline=box_color, width=3)

        # Draw name under box
        text_x = left
        text_y = bottom + 4

        bbox = draw.textbbox((0, 0), name, font=FONT)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]

        # Clamp text inside image
        text_x = max(0, min(text_x, pil_image.width - text_w - 4))
        text_y = max(0, min(text_y, pil_image.height - text_h - 4))
        
        draw.rectangle(((text_x, text_y), (text_x + text_w + 4, text_y + text_h + 4)), fill="green")
        text_fill = "white" if box_color == "green" else "black"
        draw.text((text_x + 2, text_y + 2), name, fill=text_fill, font=FONT)


        print(f"  Face {idx+1}: {name}")

    # Save annotated image once
    #output_path = img_path.with_name(f"{img_path.stem}_annotated.jpg")
    output_path = OUTPUT_DIR / f"{img_path.stem}_annotated{img_path.suffix}"
    pil_image.save(output_path, quality=95)
    print(f"Saved annotated image to: {output_path}")

    # Display in Jupyter
    display(pil_image)


In [ ]:
# Assume faces_with_people is already row-wise sorted
ordered_list_path = img_path.with_name(f"{img_path.stem}_faces_order.txt")

def sort_faces_rowwise(faces_with_people, row_threshold=30):
    """
    Sort faces top-to-bottom, then left-to-right within each row.
    faces_with_people: list of (box, person) tuples
    row_threshold: vertical distance to consider as same row (pixels)
    """
    # Compute the vertical center of each face
    faces_with_people = [
        (box, person, (box[0] + box[2]) // 2)  # center_y
        for box, person in faces_with_people
    ]

    # Sort by vertical center first
    faces_with_people.sort(key=lambda x: x[2])  # sort by center_y

    # Group into rows
    rows = []
    current_row = []
    for box, person, center_y in faces_with_people:
        if not current_row:
            current_row.append((box, person))
            last_center_y = center_y
            continue
        # If within threshold, same row
        if abs(center_y - last_center_y) <= row_threshold:
            current_row.append((box, person))
        else:
            # Sort current row left-to-right
            current_row.sort(key=lambda x: x[0][3])  # left
            rows.append(current_row)
            current_row = [(box, person)]
        last_center_y = center_y

    # Add last row
    if current_row:
        current_row.sort(key=lambda x: x[0][3])
        rows.append(current_row)

    # Flatten list
    sorted_faces = [fp for row in rows for fp in row]
    return sorted_faces

faces_with_people = list(zip(face_locations, people))

# Sort faces row-wise (top → bottom, left → right)
faces_with_people = sort_faces_rowwise(faces_with_people, row_threshold=40)

# Print order
print("Faces in row-wise order (top→bottom, left→right):")

with open(ordered_list_path, "w", encoding="utf-8") as f:
    #f.write("Faces in row-wise order (top→bottom, left→right):\n")
    for idx, (box, person) in enumerate(faces_with_people):
        name = person.get("name", "Unknown")
        top, right, bottom, left = box
        #f.write(f"{idx+1}: {name} \n")
        f.write(f"{name}, ")
        print(f"{idx+1}: {name} ")
        #f.write(f"{idx+1}: {name} (Top={top}, Left={left})\n")
        #print(f"{idx+1}: {name} (Top={top}, Left={left})")


In [ ]:
# Assume faces_with_people is already row-wise sorted
ordered_list_path = img_path.with_name(f"{img_path.stem}_faces_order.txt")

with open(ordered_list_path, "w", encoding="utf-8") as f:
    f.write("Faces in row-wise order (top→bottom, left→right):\n")
    for idx, (box, person) in enumerate(faces_with_people):
        name = person.get("name", "Unknown")
        top, right, bottom, left = box
        f.write(f"{idx+1}: {name} (Top={top}, Left={left})\n")

print(f"Saved ordered face list to: {ordered_list_path}")


In [ ]:
import os
import shutil

def sort_images(folder_path):
    image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp")

    for filename in os.listdir(folder_path):
        full_path = os.path.join(folder_path, filename)

        if not os.path.isfile(full_path):
            continue

        if not filename.lower().endswith(image_extensions):
            continue

        # EXACT rule: full stem, replace underscores only
        name_no_ext = filename.rsplit(".", 1)[0]
        folder_name = name_no_ext.replace("_", " ")

        dest_folder = os.path.join(folder_path, folder_name)
        os.makedirs(dest_folder, exist_ok=True)

        shutil.move(full_path, os.path.join(dest_folder, filename))

        print(f"{filename} → {folder_name}")

# USE YOUR REAL PATH HERE
sort_images("WarnersBayYear12/faces")